In [148]:
import pandas as pd

In [149]:
df = pd.read_csv('./results/raw_shipment_classification_dataset.csv')

In [150]:
df.duplicated().sum()

np.int64(1457)

In [151]:
df.drop_duplicates(inplace=True)

In [152]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 967 entries, 0 to 2423
Data columns (total 37 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   bill_id                           967 non-null    int64  
 1   bill_id_dup                       967 non-null    int64  
 2   vendor_id                         967 non-null    int64  
 3   vendor_name                       967 non-null    object 
 4   bni_created_time                  967 non-null    object 
 5   delay_days                        967 non-null    int64  
 6   shipped_dt                        967 non-null    object 
 7   eta_dt                            967 non-null    object 
 8   po_date_dt                        967 non-null    object 
 9   receipt_dt                        847 non-null    object 
 10  shipment_days                     967 non-null    float64
 11  promised_transit_days             967 non-null    float64
 12  days_until_e

In [153]:
date_time_columns = ['shipped_dt', 'eta_dt']
for col in date_time_columns:
    df[f'{col}'] = pd.to_datetime(df[f'{col}'], errors='raise')

df['shipped_date_weekday'] = df['shipped_dt'].dt.weekday
df['shipped_date_month'] = df['shipped_dt'].dt.month
df['shipped_date_day'] = df['shipped_dt'].dt.day

df.drop(columns=date_time_columns, inplace=True)

In [154]:
df['total_bcy']

0       USD 110,565.000
4       USD 157,308.480
20      USD 174,420.540
22      USD 174,420.540
28      USD 174,420.540
             ...       
2417    USD 165,960.000
2419    USD 121,680.000
2420    USD 121,680.000
2421    USD 165,960.000
2423    USD 155,142.000
Name: total_bcy, Length: 967, dtype: object

In [155]:
# 1. Remove 'USD' and any surrounding whitespace
df['total_bcy'] = df['total_bcy'].str.replace('USD', '').str.strip()

# 2. FIX: Remove all thousands separators (commas)
df['total_bcy'] = df['total_bcy'].str.replace(',', '')

# 3. Convert the clean string to float
df['total_bcy'] = df['total_bcy'].astype(float)

In [156]:
import numpy as np
import pandas as pd

numeric_cols = [
    "promised_transit_days",
    "days_until_eta",
    "days_since_ship_so_far",
    "lead_time_days",
    "tariff_amount",
    "ocean_freight",
    "total_bcy",
    "quantity_in",
    "vendor_avg_promised_transit_days",
    "vendor_p50_promised_transit_days",
    "vendor_p90_promised_transit_days",
    "vendor_avg_realized_delay_days",
    "vendor_p50_realized_delay_days",
    "vendor_p90_realized_delay_days",
    "vendor_on_time_rate",
    "vendor_shipments_with_receipt",
]

df = df.copy()

# 1) Split days_until_eta into two features (keep signal for overdue vs remaining)
# df["days_to_eta"]   = np.clip(df["days_until_eta"], a_min=0, a_max=None)          # time remaining
# df["days_overdue"]  = np.clip(-df["days_until_eta"], a_min=0, a_max=None)         # overdue days
# df.drop(columns=["days_until_eta"], inplace=True)

# 2) Columns that must be non-negative: clip & flag corrections
# nonneg_cols = [
#     "promised_transit_days",
#     "days_since_ship_so_far",
#     "lead_time_days",
#     "tariff_amount",
#     "ocean_freight",
#     "total_bcy",
#     "quantity_in",
#     "vendor_avg_promised_transit_days",
#     "vendor_p50_promised_transit_days",
#     "vendor_p90_promised_transit_days",
#     "vendor_avg_realized_delay_days",
#     "vendor_p50_realized_delay_days",
#     "vendor_p90_realized_delay_days",
#     "vendor_shipments_with_receipt",
# ]

# for col in nonneg_cols:
#     flag_col = f"{col}__was_negative"
#     df[flag_col] = (df[col] < 0).astype("uint8")
#     df[col] = np.clip(df[col], a_min=0, a_max=None)

# 3) Probability/rate column: cap to [0, 1] and flag out-of-range
# df["vendor_on_time_rate__was_oob"] = (
#     (df["vendor_on_time_rate"] < 0) | (df["vendor_on_time_rate"] > 1)
# ).astype("uint8")
# df["vendor_on_time_rate"] = df["vendor_on_time_rate"].clip(lower=0, upper=1)

# 4) Optional: log1p transform for skewed monetary/quantity/count features (use for linear models)
# log1p_cols = ["tariff_amount", "ocean_freight", "total_bcy", "quantity_in", "vendor_shipments_with_receipt"]
# for col in log1p_cols:
#     df[f"{col}__log1p"] = np.log1p(df[col])  # keep original too (trees like raw scale)

    

In [157]:
# Convert total_bcy to numeric
df['total_bcy'] = pd.to_numeric(df['total_bcy'], errors='coerce')


In [158]:
distance_dict = {
    'INDIA': 11000,
    'CHINA': 6000,
    'INDONESIA': 8200,
    'VIETNAM': 6200,
    'ECUADOR': 2100,
    'THAILAND': 8100
}
df['coo'].value_counts()

coo
INDIA        810
CHINA         54
INDONESIA     49
VIETNAM       44
ECUADOR        7
THAILAND       3
Name: count, dtype: int64

In [159]:
df['distance_nm'] = df['coo'].map(distance_dict)
df.drop(columns=['coo','item_product_category','vendor_name'], inplace=True)
df.dropna(inplace=True)

In [160]:
df.columns

Index(['bill_id', 'bill_id_dup', 'vendor_id', 'bni_created_time', 'delay_days',
       'po_date_dt', 'receipt_dt', 'shipment_days', 'promised_transit_days',
       'days_until_eta', 'days_since_ship_so_far', 'lead_time_days', 'scac',
       'tariff_amount', 'ocean_freight', 'delivery_terms', 'po_shipment_terms',
       'tariff_type', 'total_bcy', 'quantity_in', 'item_sku', 'item_brand',
       'item_manufacturer', 'item_size', 'vendor_avg_promised_transit_days',
       'vendor_p50_promised_transit_days', 'vendor_p90_promised_transit_days',
       'vendor_avg_realized_delay_days', 'vendor_p50_realized_delay_days',
       'vendor_p90_realized_delay_days', 'vendor_on_time_rate',
       'vendor_shipments_with_receipt', 'shipped_date_weekday',
       'shipped_date_month', 'shipped_date_day', 'distance_nm'],
      dtype='object')

In [161]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 821 entries, 52 to 2415
Data columns (total 36 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   bill_id                           821 non-null    int64  
 1   bill_id_dup                       821 non-null    int64  
 2   vendor_id                         821 non-null    int64  
 3   bni_created_time                  821 non-null    object 
 4   delay_days                        821 non-null    int64  
 5   po_date_dt                        821 non-null    object 
 6   receipt_dt                        821 non-null    object 
 7   shipment_days                     821 non-null    float64
 8   promised_transit_days             821 non-null    float64
 9   days_until_eta                    821 non-null    float64
 10  days_since_ship_so_far            821 non-null    float64
 11  lead_time_days                    821 non-null    float64
 12  scac       

In [162]:
df.to_csv('./results/cleaned_shipment_classification_dataset.csv', index=False)

In [163]:
# df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
label_encoder = LabelEncoder()

# Ensure categorical features are strings (clean)
categorical_cols = [
    'scac', 'delivery_terms',
    'po_shipment_terms', 'tariff_type', 'item_brand',
    'item_manufacturer', 
    'item_size'
]

for col in categorical_cols:
    df_encoded[col] = df_encoded[col].astype(str).str.strip().replace('', 'Unknown')

for col in categorical_cols:
    df_encoded[col] = label_encoder.fit_transform(df_encoded[col].astype(str))


In [164]:
from sklearn.preprocessing import StandardScaler

columns_to_scale = [
    "tariff_amount",
    "quantity_in",
    "ocean_freight",
]

scaler = StandardScaler()
df_encoded[columns_to_scale] = scaler.fit_transform(df_encoded[columns_to_scale])